# Mimi → Wav2Vec2 Bridge Module Training

**Teacher model**: `facebook/wav2vec2-base-960h` (768-dim, 25 Hz after down-interpolation)  
**Student model**: Mimi discrete tokens (12.5 Hz) → bridge → 768-dim continuous features at 25 Hz  
**Training target**: Minimise distance between bridge output and real Wav2Vec2 hidden states


## 1. Clone Repository

In [ ]:
!git clone https://github.com/MoshiHead/moshi-wav2vec-bridge-v2.git moshi-wav2vec-bridge-v2

In [ ]:
!mkdir -p /workspace/moshi-wav2vec-bridge-v2/data/LibriSpeech

# Download with progress bar
!aria2c -x 16 -s 16 https://www.openslr.org/resources/12/train-clean-100.tar.gz

# Extract quietly (no file listing)
!tar -xf train-clean-100.tar.gz

# Move extracted folder
!mv LibriSpeech/train-clean-100 /workspace/moshi-wav2vec-bridge-v2/data/LibriSpeech

# Clean up
!rm -rf train-clean-100.tar.gz LibriSpeech

print("✅ Done! Data is in /workspace/moshi-wav2vec-bridge-v2/data/")

In [ ]:
%cd moshi-wav2vec-bridge-v1

## 2. Install Dependencies

> **Note**: `onnxruntime-gpu` is no longer needed (HuBERT ONNX replaced by Wav2Vec2 HuggingFace).  
> `transformers>=4.40.0` is required for `Wav2Vec2Model`.

In [ ]:
!pip install -q moshi safetensors huggingface_hub transformers>=4.40.0 torchaudio librosa tqdm pyworld

In [ ]:
!pip install -r requirements.txt

## 3. System dependencies (RunPod / Colab)

In [ ]:
!apt-get update -qq && apt-get install -y aria2 unzip zstd

## 4. Download Dataset (LibriSpeech subset)

In [ ]:
#_aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ

In [ ]:
# from huggingface_hub import snapshot_download, login
# import os

# # hf_token = ""  # replace with your token
# # login(token=hf_token)

# repo_id     = "Darknsu/librispeech-dataset-small-part"
# download_dir = "/workspace/moshi-wav2vec-bridge-v1"
# os.makedirs(download_dir, exist_ok=True)

# print("📥 Downloading dataset...")
# snapshot_download(
#     repo_id=repo_id,
#     repo_type="dataset",
#     local_dir=download_dir,
#     local_dir_use_symlinks=False,
# )
# print("✅ Downloaded to:", download_dir)

In [ ]:
# # Extract the dataset archive
# !tar -I zstd -xf librispeech_10000_audio.tar.zst

## 5. Pre-warm Wav2Vec2 Model Download

`facebook/wav2vec2-base-960h` will be auto-downloaded by HuggingFace on first use.  
Run this cell to pre-download it before training so workers don't race:

In [ ]:
# Pre-download facebook/wav2vec2-base-960h to HuggingFace cache
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

print("⬇️  Downloading facebook/wav2vec2-base-960h ...")
_ = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
_ = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
print("✅ Model cached in HuggingFace cache (~360 MB)")

## 6. Verify config.yaml

Key settings that must match your setup:

In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

print(f"output_dim      : {cfg['model']['output_dim']}   (should be 768)")
print(f"wav2vec2_model  : {cfg['paths']['wav2vec2_model']}")
print(f"mimi_model      : {cfg['paths']['mimi_model']}")
print(f"upsample_factor : {cfg['model']['upsample_factor']}   (should be 2)")
print(f"batch_size      : {cfg['training']['batch_size']}")
print(f"num_epochs      : {cfg['training']['num_epochs']}")

## 7. Pre-extract & Cache Features

Runs `preprocess.py` across all GPUs in parallel.  
Each worker:
1. Loads audio → Mimi encoder → tokens `(T_m, 8)` at 12.5 Hz  
2. Loads audio → `facebook/wav2vec2-base-960h` → last_hidden_state  
   → down-interpolates to 25 Hz → features `(T_h, 768)`  
3. Saves both as `.pt` files in `data/cache/`

In [ ]:
!torchrun --nproc_per_node=4 preprocess.py \
    --dataset librispeech \
    --root ./data/LibriSpeech/train-clean-100 \
    --out_dir data \
    --val_frac 0.1 \
    --preextract \
    --device cuda \
    --num_workers 16

## 8. (Optional) Archive Preprocessed Cache

Save the preprocessed cache so you don't re-run extraction if you restart:

In [ ]:
!tar -I 'zstd -1' -cf librispeech_10000_audio_preprocess_wav2vec2.tar.zst ./data

## 9. Train the Bridge Model

Multi-GPU DDP training. Adjust `--nproc_per_node` to match your GPU count.

In [ ]:
!torchrun --nproc_per_node=4 train.py --config config.yaml

## 10. Inference

Run the bridge model on a single audio file.

In [ ]:
!python inference.py \
    --audio /path/to/test.wav \
    --checkpoint checkpoints/bridge_best.pt \
    --config config.yaml \
    --device cuda \
    --output bridge_output.npy

# Expected output shape: (T, 768)  at 25 Hz
# T = ceil(audio_length_seconds * 25)

## 11. Compare Bridge vs Wav2Vec2 Ground-Truth

Side-by-side quality metrics: MSE, cosine similarity, SNR.

In [ ]:
!python compare_inference.py \
    --audio /path/to/test.wav \
    --checkpoint checkpoints/bridge_best.pt \
    --config config.yaml \
    --device cuda \
    --plot

# Saves automatically:
#   bridge_pred_features.npy   — bridge output   (T, 768)
#   wav2vec2_gt_features.npy   — real wav2vec2    (T, 768)